<a href="https://colab.research.google.com/github/Giraffe-Shin/trading/blob/main/Basic/Basic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
from google.colab import userdata

# 기존 설정 재사용
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = 'YOUR_TOKEN_HERE'

username = 'Giraffe-Shin'
repo = 'trading'
branch = 'main' # 이 부분은 contents API에서는 직접적으로 사용되지 않지만, 다른 작업 시 필요할 수 있습니다.
directory_path = 'data'

# GitHub API를 사용하여 디렉토리 내용 가져오기
api_url = f'https://api.github.com/repos/{username}/{repo}/contents/{directory_path}'

headers = {
    'Authorization': f'token {github_token}',
    'Accept': 'application/vnd.github.v3+json' # GitHub API 버전 3 사용 명시
}

try:
    response = requests.get(api_url, headers=headers)

    if response.status_code == 200:
        contents = response.json()
        file_names = [item['name'] for item in contents if item['type'] == 'file']
        print(f"'{directory_path}' 디렉토리의 파일 목록:")
        for file_name in file_names:
            print(f"- {file_name}")
    elif response.status_code == 401:
        print("인증 실패: 토큰이 유효하지 않거나 권한이 없습니다. GitHub 토큰을 확인해주세요.")
    elif response.status_code == 404:
        print(f"디렉토리 '{directory_path}'를 찾을 수 없습니다. 경로를 확인해주세요.")
    else:
        print(f"파일 목록을 가져오는 데 실패했습니다. 상태 코드: {response.status_code}, 메시지: {response.text}")
except Exception as e:
    print(f"오류 발생: {e}")


'data' 디렉토리의 파일 목록:
- germany_actual_generation.csv
- germany_cbet_trading.csv
- germany_cbpf_physical_flow.csv
- germany_energy_master.csv
- germany_forecast_day_ahead.csv
- germany_market_prices.csv
- germany_renewable_share_daily.csv


In [3]:
import pandas as pd
import requests
from io import StringIO
from google.colab import userdata

# --- 필수 설정 ---
try:
    # Colab 보안 비밀(Secrets)에서 토큰을 가져옵니다.
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = 'YOUR_TOKEN_HERE'

username = 'Giraffe-Shin'
repo = 'trading'
branch = 'main'
file_path = 'data/germany_actual_generation.csv'

# API를 통한 프라이빗 파일 접근 URL
url = f'https://raw.githubusercontent.com/{username}/{repo}/{branch}/{file_path}'

headers = {'Authorization': f'token {github_token}'}

try:
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        print("성공적으로 파일을 불러왔습니다!")
        display(df.head())
    elif response.status_code == 401:
        print("인증 실패: 토큰이 유효하지 않거나 권한이 없습니다.")
    elif response.status_code == 404:
        print("파일을 찾을 수 없습니다(404). 토큰 이름, 경로, 브랜치명을 다시 확인해주세요.")
    else:
        print(f"실패했습니다. 상태 코드: {response.status_code}")
except Exception as e:
    print(f"오류 발생: {e}")

성공적으로 파일을 불러왔습니다!


,timestamp,Hydro pumped storage consumption,Cross border electricity trading,Nuclear,Hydro Run-of-River,Biomass,Fossil brown coal / lignite,Fossil hard coal,Fossil oil,Fossil coal-derived gas,...,Hydro pumped storage,Others,Waste,Wind offshore,Wind onshore,Solar,Load,Residual load,Renewable share of load,Renewable share of generation
0,2022-12-31 23:00:00,-1975.8,-12645.2,2460.7,1627.7,4022.8,3845.6,1825.5,306.0,670.0,...,127.0,187.4,1050.3,2739.0,27260.9,0.0,38691.8,8691.8,93.7,75.5
1,2022-12-31 23:15:00,-2009.5,-12544.0,2458.5,1624.8,4012.0,3847.7,1828.6,306.8,648.2,...,180.1,187.3,1049.1,2975.6,27311.5,0.0,38374.2,8087.1,95.1,75.7
2,2022-12-31 23:30:00,-2244.0,-12526.2,2459.6,1624.8,4010.0,3860.2,1825.2,306.9,648.2,...,106.9,187.2,1055.2,3269.6,27699.4,0.0,38248.0,7279.0,97.2,76.0
3,2022-12-31 23:45:00,-2077.3,-12515.4,2457.9,1622.0,4004.0,3861.6,1819.7,306.0,639.1,...,87.0,187.3,1043.6,3259.5,27170.0,0.0,38070.2,7640.6,96.3,75.8
4,2023-01-01 00:00:00,-1204.1,-15410.1,2457.7,1622.3,3988.4,3869.8,1811.4,306.0,635.0,...,547.7,187.3,1047.1,3144.1,27853.6,0.0,37733.4,6735.7,98.5,76.3


In [4]:
# 데이터프레임의 컬럼 이름 리스트 출력
column_list = df.columns.tolist()
print(column_list)

['timestamp', 'Hydro pumped storage consumption', 'Cross border electricity trading', 'Nuclear', 'Hydro Run-of-River', 'Biomass', 'Fossil brown coal / lignite', 'Fossil hard coal', 'Fossil oil', 'Fossil coal-derived gas', 'Fossil gas', 'Geothermal', 'Hydro water reservoir', 'Hydro pumped storage', 'Others', 'Waste', 'Wind offshore', 'Wind onshore', 'Solar', 'Load', 'Residual load', 'Renewable share of load', 'Renewable share of generation']


In [5]:
# 제외할 컬럼 리스트
cols_to_exclude = [
    "Load",
    "Residual load",
    "Renewable share of generation",
    "Renewable share of load",
    "Cross border electricity trading"
]

# 해당 컬럼들을 제외하고 데이터프레임 업데이트
df = df.drop(columns=cols_to_exclude)

# 결과 확인
print("남은 컬럼 목록:")
print(df.columns.tolist())
display(df.head())

남은 컬럼 목록:
['timestamp', 'Hydro pumped storage consumption', 'Nuclear', 'Hydro Run-of-River', 'Biomass', 'Fossil brown coal / lignite', 'Fossil hard coal', 'Fossil oil', 'Fossil coal-derived gas', 'Fossil gas', 'Geothermal', 'Hydro water reservoir', 'Hydro pumped storage', 'Others', 'Waste', 'Wind offshore', 'Wind onshore', 'Solar']


,timestamp,Hydro pumped storage consumption,Nuclear,Hydro Run-of-River,Biomass,Fossil brown coal / lignite,Fossil hard coal,Fossil oil,Fossil coal-derived gas,Fossil gas,Geothermal,Hydro water reservoir,Hydro pumped storage,Others,Waste,Wind offshore,Wind onshore,Solar
0,2022-12-31 23:00:00,-1975.8,2460.7,1627.7,4022.8,3845.6,1825.5,306.0,670.0,1882.0,18.2,73.1,127.0,187.4,1050.3,2739.0,27260.9,0.0
1,2022-12-31 23:15:00,-2009.5,2458.5,1624.8,4012.0,3847.7,1828.6,306.8,648.2,1880.9,18.5,65.9,180.1,187.3,1049.1,2975.6,27311.5,0.0
2,2022-12-31 23:30:00,-2244.0,2459.6,1624.8,4010.0,3860.2,1825.2,306.9,648.2,1893.2,18.7,65.4,106.9,187.2,1055.2,3269.6,27699.4,0.0
3,2022-12-31 23:45:00,-2077.3,2457.9,1622.0,4004.0,3861.6,1819.7,306.0,639.1,1885.9,18.6,83.6,87.0,187.3,1043.6,3259.5,27170.0,0.0
4,2023-01-01 00:00:00,-1204.1,2457.7,1622.3,3988.4,3869.8,1811.4,306.0,635.0,1696.4,18.6,57.3,547.7,187.3,1047.1,3144.1,27853.6,0.0


In [6]:
# 'timestamp'를 제외한 모든 수치형 열의 합계를 구하여 새로운 'Total' 열 생성
numeric_cols = df.columns.drop('timestamp')
df['Total'] = df[numeric_cols].sum(axis=1)

# 결과 확인
display(df[['timestamp'] + list(numeric_cols) + ['Total']].head())

,timestamp,Hydro pumped storage consumption,Nuclear,Hydro Run-of-River,Biomass,Fossil brown coal / lignite,Fossil hard coal,Fossil oil,Fossil coal-derived gas,Fossil gas,Geothermal,Hydro water reservoir,Hydro pumped storage,Others,Waste,Wind offshore,Wind onshore,Solar,Total
0,2022-12-31 23:00:00,-1975.8,2460.7,1627.7,4022.8,3845.6,1825.5,306.0,670.0,1882.0,18.2,73.1,127.0,187.4,1050.3,2739.0,27260.9,0.0,46120.4
1,2022-12-31 23:15:00,-2009.5,2458.5,1624.8,4012.0,3847.7,1828.6,306.8,648.2,1880.9,18.5,65.9,180.1,187.3,1049.1,2975.6,27311.5,0.0,46386.0
2,2022-12-31 23:30:00,-2244.0,2459.6,1624.8,4010.0,3860.2,1825.2,306.9,648.2,1893.2,18.7,65.4,106.9,187.2,1055.2,3269.6,27699.4,0.0,46786.5
3,2022-12-31 23:45:00,-2077.3,2457.9,1622.0,4004.0,3861.6,1819.7,306.0,639.1,1885.9,18.6,83.6,87.0,187.3,1043.6,3259.5,27170.0,0.0,46368.5
4,2023-01-01 00:00:00,-1204.1,2457.7,1622.3,3988.4,3869.8,1811.4,306.0,635.0,1696.4,18.6,57.3,547.7,187.3,1047.1,3144.1,27853.6,0.0,48038.6
